# EdgeGuard campaign control
Thin wrapper for initialization, planning, verified reuse, and the data-preparation handoff.

## Compact campaign status

Each stage prints a compact overview, periodic progress rows, and a post-stage summary. Full logs remain in files. Failures produce an `edgeguard-failure-<campaign>-<stage>.zip` with environment, state, artifact identities, recovery state, traceback, and log tails.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("EDGEGUARD_PROJECT_ROOT", ".")).resolve()
CAMPAIGN_ROOT = Path(os.environ.get("EDGEGUARD_CAMPAIGN_ROOT", "/content/edgeguard-campaign"))
PROJECT_COMMIT = os.environ.get("EDGEGUARD_PROJECT_COMMIT", "")
CAMPAIGN_ID = os.environ.get("EDGEGUARD_CAMPAIGN_ID", "eg-colab-campaign")
PROFILE = os.environ.get("EDGEGUARD_CAMPAIGN_PROFILE", "colab")
AUTO_CONTINUE = os.environ.get("EDGEGUARD_AUTO_CONTINUE", "0") == "1"
actual = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if len(PROJECT_COMMIT) != 40 or actual != PROJECT_COMMIT:
    raise RuntimeError("exact project commit mismatch")
base = [sys.executable, "-m", "edgeguard.campaign", "--repository", str(PROJECT_ROOT)]

In [ ]:
if not (CAMPAIGN_ROOT / "campaign_manifest.json").exists():
    subprocess.run(
        base
        + [
            "init",
            "--campaign-root",
            str(CAMPAIGN_ROOT),
            "--campaign-id",
            CAMPAIGN_ID,
            "--profile",
            PROFILE,
        ],
        check=True,
    )
plan = subprocess.run(
    base + ["plan", "--campaign-root", str(CAMPAIGN_ROOT)],
    check=True,
    capture_output=True,
    text=True,
)
print(json.dumps(json.loads(plan.stdout), indent=2))
if AUTO_CONTINUE:
    subprocess.run(
        base + ["run", "--campaign-root", str(CAMPAIGN_ROOT), "--stop-after", "dataset_prepare"],
        check=True,
    )
subprocess.run(
    base + ["report", "--campaign-root", str(CAMPAIGN_ROOT), "--audience", "assistant"], check=True
)